# 03 – Baselines

The numbers every later model has to beat. Three baselines, all trained on `train` and scored on `valid`:

0. **No model:** predict the training stop rate (≈ 0.20) for every trial.
1. **Logistic regression on 2 features:** `phase` + `lead_sponsor_class`.
2. **Logistic regression on all 38 features.**

The **test set is not used here**. It's scored once, in notebook 05.

## 0. Load `analysis.features`

In [1]:
from sqlalchemy import text
import pandas as pd
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))  
from src.db import engine




query= text("""
    SELECT * 
    FROM analysis.features
""")
with engine.connect() as conn:
    df=pd.read_sql(query, conn)


### Check: every cohort trial is loaded (expect 47,210 rows × 44 columns)

In [2]:
df.shape

(47210, 44)

In [3]:
df['label'].value_counts()

label
0.0    25420
1.0     6483
Name: count, dtype: int64

## 1. Keep the labelled trials only

Trials excluded from the main analysis (unknown status, still ongoing, …) have `label` = NaN. That leaves **31,903 trials**.

In [4]:
filtered=df.dropna(subset=["label"])
print(filtered.shape)
filtered.head()

(31903, 44)


,nct_id,label,label_sensitivity,split,registration_date,registration_year,status_2018,phase,number_of_arms,has_dmc,...,is_drug,is_device,is_biological,is_procedure,is_behavioral,is_radiation,is_dietary_supplement,is_other_intervention,n_conditions,is_oncology
0,NCT03703765,0.0,0.0,test,2018-01-18,2018,RECRUITING,N/A,1.0,yes,...,0,1,0,0,0,0,0,0,1,0
1,NCT03698591,0.0,0.0,test,2018-10-04,2018,NOT_YET_RECRUITING,N/A,2.0,yes,...,0,1,0,0,0,0,0,0,2,0
3,NCT03698565,0.0,0.0,test,2018-09-24,2018,NOT_YET_RECRUITING,N/A,1.0,no,...,0,0,0,0,0,0,0,1,1,0
4,NCT03698552,1.0,1.0,test,2018-09-26,2018,RECRUITING,Phase 1/Phase 2,1.0,no,...,0,0,1,0,0,0,0,0,5,1
6,NCT03698513,0.0,0.0,test,2018-10-04,2018,RECRUITING,Phase 1,9.0,no,...,1,0,0,0,0,0,0,0,1,0


### Column types

In [5]:
filtered.dtypes

nct_id                              str
label                           float64
label_sensitivity               float64
split                               str
registration_date                object
registration_year                 int64
status_2018                         str
phase                               str
number_of_arms                  float64
has_dmc                             str
allocation                          str
masking                             str
primary_purpose                     str
intervention_model                  str
enrollment_target               float64
planned_duration_months         float64
months_registration_to_start    float64
overdue_2018                    float64
gender                              str
accepts_healthy_volunteers      float64
min_age_years                   float64
max_age_years                   float64
no_min_age                        int64
no_max_age                        int64
n_sites                           int64


## 2. Feature lists

**38 features** in four groups. The other 6 columns (IDs, label, split, registration date) are not features.

In [6]:
non_features = [
    "nct_id",
    "label",
    "label_sensitivity",
    "split",
    "registration_date",
    "registration_year",
]

categorical_features = [
    "status_2018",
    "phase",
    "has_dmc",
    "allocation",
    "masking",
    "primary_purpose",
    "intervention_model",
    "gender",
    "lead_sponsor_class",
]

numeric_features = [
    "enrollment_target",
    "number_of_arms",
    "planned_duration_months",
    "months_registration_to_start",
    "min_age_years",
    "max_age_years",
    "n_sites",
    "n_countries",
    "n_collaborators",
    "n_interventions",
    "n_intervention_types",
    "n_conditions",
]

binary_features = [
    "overdue_2018",
    "accepts_healthy_volunteers",
    "no_min_age",
    "no_max_age",
    "no_sites_listed",
    "single_site",
    "has_us_site",
    "had_country_removed",
    "is_drug",
    "is_device",
    "is_biological",
    "is_procedure",
    "is_behavioral",
    "is_radiation",
    "is_dietary_supplement",
    "is_other_intervention",
    "is_oncology",
]

all_features = categorical_features + numeric_features + binary_features

### Skewed columns get a log transform

In [7]:
# The sizes and counts are heavily skewed (enrolment up to 2,000,000, sites up to 1,457),
# so they get a log transform. The other numeric columns can be negative or are already on
# a reasonable scale, so they're only scaled.
log_numeric_features = [
    "enrollment_target",
    "n_sites",
    "n_countries",
    "n_collaborators",
    "n_interventions",
    "n_conditions",
]
plain_numeric_features = [c for c in numeric_features if c not in log_numeric_features]

assert len(all_features) == 38
assert set(log_numeric_features) | set(plain_numeric_features) == set(numeric_features)
print(f"{len(categorical_features)} categorical, {len(log_numeric_features)} log-numeric, "
      f"{len(plain_numeric_features)} numeric, {len(binary_features)} binary")

9 categorical, 6 log-numeric, 6 numeric, 17 binary


### Split sizes and stop rates

These should match the SQL: 11,776 / 9,157 / 10,970 trials, each about 20% stopped early.

In [8]:
filtered["split"].value_counts()

split
train    11776
test     10970
valid     9157
Name: count, dtype: int64

In [9]:
filtered.groupby("split")["label"].agg('mean')

split
test     0.198086
train    0.208984
valid    0.201922
Name: label, dtype: float64

## 3. Train and validation sets

**The test set is not built in this notebook.** It's scored exactly once, in notebook 05, so that no decision made here can be tuned to it.

In [10]:
# Only train and valid are built here. The test set stays untouched until notebook 05.
train = filtered[filtered["split"] == "train"]
valid = filtered[filtered["split"] == "valid"]

X_train, y_train = train[all_features], train["label"]
X_valid, y_valid = valid[all_features], valid["label"]

print(f"train: {len(X_train):,} trials, {y_train.mean():.1%} stopped early")
print(f"valid: {len(X_valid):,} trials, {y_valid.mean():.1%} stopped early")

train: 11,776 trials, 20.9% stopped early
valid: 9,157 trials, 20.2% stopped early


### Missing values in the training features

In [11]:
missing_train = X_train.isna().sum()
display(missing_train[missing_train > 0].sort_values(ascending=False))

max_age_years                   6054
min_age_years                    660
number_of_arms                    70
planned_duration_months           10
months_registration_to_start       9
enrollment_target                  7
accepts_healthy_volunteers         5
overdue_2018                       2
dtype: int64

### Number of categories per text column

In [12]:
display(X_train[categorical_features].nunique())

status_2018            2
phase                  8
has_dmc                3
allocation             4
masking                6
primary_purpose       10
intervention_model     6
gender                 3
lead_sponsor_class     4
dtype: int64

## 4. Preprocessing

One `ColumnTransformer` (built by `make_preprocessor` in `src/preprocess.py`) handles all four column groups:

| Group | Steps |
|---|---|
| categorical (9) | fill missing with `"missing"` → one-hot encode (categories unseen in train are ignored) |
| log-numeric (6) | median fill → `log(1 + x)` → scale |
| numeric (6) | median fill → scale |
| binary (17) | fill the few missing values with the most frequent value |

Everything is **learned from train only** (`fit_transform` on train, `transform` on valid). In the model sections, the same preprocessor goes inside a `Pipeline` with the model, so a single `.fit()` does both.

The cell below runs it on its own once, just to check the output: no missing values, and the same columns in train and valid.

In [13]:
from src.preprocess import make_preprocessor

preprocessor = make_preprocessor(
    categorical=categorical_features,
    log_numeric=log_numeric_features,
    numeric=plain_numeric_features,
    binary=binary_features,
)
# Return DataFrames with column names, so the output is easy to inspect
preprocessor.set_output(transform="pandas")

# fit on train only; valid is only transformed with what was learned from train
X_train_prepared = preprocessor.fit_transform(X_train)
X_valid_prepared = preprocessor.transform(X_valid)

print(f"train: {X_train_prepared.shape}, valid: {X_valid_prepared.shape}")
assert X_train_prepared.shape[1] == X_valid_prepared.shape[1]
assert list(X_train_prepared.columns) == list(X_valid_prepared.columns)
assert X_train_prepared.isna().sum().sum() == 0
assert X_valid_prepared.isna().sum().sum() == 0

train: (11776, 75), valid: (9157, 75)


### First rows after preprocessing

In [14]:
display(X_train_prepared.head())

,status_2018_NOT_YET_RECRUITING,status_2018_RECRUITING,phase_Early Phase 1,phase_N/A,phase_Phase 1,phase_Phase 1/Phase 2,phase_Phase 2,phase_Phase 2/Phase 3,phase_Phase 3,phase_Phase 4,...,had_country_removed,is_drug,is_device,is_biological,is_procedure,is_behavioral,is_radiation,is_dietary_supplement,is_other_intervention,is_oncology
183,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
933,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1091,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1697,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2874,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5. Scoring function

`evaluate()` in `src/evaluate.py` returns three scores for one model on one dataset:

| Metric | Meaning | Random guessing | Perfect |
|---|---|---|---|
| **PR-AUC** (main) | how well the model finds the stopped trials, without false alarms | ≈ 0.20 (the positive rate) | 1.0 |
| ROC-AUC | how often a stopped trial is ranked above a completed one | 0.5 | 1.0 |
| Brier | squared error of the probabilities (lower is better) | ≈ 0.16 for a constant 0.20 | 0 |

Every model is **trained on `train` and scored on `valid`**. The results are collected in `results`.

In [15]:
from src.evaluate import evaluate

results = []  # one dict per model, turned into a table at the end

## 6. Baseline 0 – no model

`DummyClassifier(strategy="prior")` ignores the features and predicts the training stop rate (≈ 0.21) for every trial. **Expected: PR-AUC ≈ 0.20 and ROC-AUC = 0.50.** If it doesn't give that, something upstream is wrong.

In [16]:
from sklearn.dummy import DummyClassifier

baseline_0 = DummyClassifier(strategy="prior")
baseline_0.fit(X_train, y_train)

valid_prob_0 = baseline_0.predict_proba(X_valid)[:, 1]
results.append(evaluate(y_valid, valid_prob_0, "0 · no model (predict the stop rate)"))
results[-1]

{'model': '0 · no model (predict the stop rate)',
 'n_trials': 9157,
 'positive_rate': 0.20192202686469368,
 'pr_auc': 0.20192202686469368,
 'roc_auc': 0.5,
 'brier': 0.16119939869273187}

## 7. Baseline 1 – logistic regression on 2 features

Only `phase` and `lead_sponsor_class`: the "simple" model promised in the README. The preprocessor and the model sit in one `Pipeline`, so `.fit()` learns the one-hot categories and the coefficients from train only.

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

baseline_1 = Pipeline([
    ("prep", make_preprocessor(categorical=["phase", "lead_sponsor_class"])),
    ("model", LogisticRegression(max_iter=1000)),
])
baseline_1.fit(X_train, y_train)

valid_prob_1 = baseline_1.predict_proba(X_valid)[:, 1]
results.append(evaluate(y_valid, valid_prob_1, "1 · logistic regression, phase + sponsor"))
results[-1]

{'model': '1 · logistic regression, phase + sponsor',
 'n_trials': 9157,
 'positive_rate': 0.20192202686469368,
 'pr_auc': 0.24786306194180377,
 'roc_auc': 0.5819143500695505,
 'brier': 0.15885368634799318}

## 8. Baseline 2 – logistic regression on all 38 features

The same pipeline shape, with the full preprocessor from section 4.

In [18]:
baseline_2 = Pipeline([
    ("prep", make_preprocessor(
        categorical=categorical_features,
        log_numeric=log_numeric_features,
        numeric=plain_numeric_features,
        binary=binary_features,
    )),
    ("model", LogisticRegression(max_iter=2000)),
])
baseline_2.fit(X_train, y_train)

valid_prob_2 = baseline_2.predict_proba(X_valid)[:, 1]
results.append(evaluate(y_valid, valid_prob_2, "2 · logistic regression, all 38 features"))
results[-1]

{'model': '2 · logistic regression, all 38 features',
 'n_trials': 9157,
 'positive_rate': 0.20192202686469368,
 'pr_auc': 0.3831334563949028,
 'roc_auc': 0.6952590240201437,
 'brier': 0.148383631524507}

## 9. Results on the validation set

The number to beat in week 3 is Baseline 2's PR-AUC. The first row shows what "no skill" looks like.

In [19]:
results_table = pd.DataFrame(results).set_index("model").round(3)
display(results_table)

,n_trials,positive_rate,pr_auc,roc_auc,brier
model,,,,,
0 · no model (predict the stop rate),9157,0.202,0.202,0.500,0.161
"1 · logistic regression, phase + sponsor",9157,0.202,0.248,0.582,0.159
"2 · logistic regression, all 38 features",9157,0.202,0.383,0.695,0.148


## 10. What did Baseline 2 learn? The largest coefficients

Every input was scaled (or is 0/1), so coefficient sizes are roughly comparable. **Positive** means the feature pushes towards *stopped early*; **negative** means towards *completed*.

This is a sanity check, not proof of cause: related features share credit (e.g. `n_sites` and `single_site`). What to look for is whether the signs make sense, and that no single feature is suspiciously strong (which would suggest leakage).

In [20]:
coefficients = pd.Series(
    baseline_2.named_steps["model"].coef_[0],
    index=baseline_2.named_steps["prep"].get_feature_names_out(),
).sort_values()

print("Towards COMPLETED (most negative):")
display(coefficients.head(10).round(3))
print("Towards STOPPED EARLY (most positive):")
display(coefficients.tail(10).round(3))

Towards COMPLETED (most negative):


status_2018_RECRUITING                     -1.043
is_behavioral                              -0.904
lead_sponsor_class_U.S. Fed                -0.599
primary_purpose_Health Services Research   -0.537
phase_Early Phase 1                        -0.528
has_dmc_unknown                            -0.424
intervention_model_Factorial Assignment    -0.345
overdue_2018                               -0.284
lead_sponsor_class_NIH                     -0.251
n_sites                                    -0.218
dtype: float64

Towards STOPPED EARLY (most positive):


intervention_model_Parallel Assignment    0.221
primary_purpose_Diagnostic                0.221
is_device                                 0.243
is_drug                                   0.243
months_registration_to_start              0.279
has_us_site                               0.282
phase_Phase 1/Phase 2                     0.290
primary_purpose_Screening                 0.300
lead_sponsor_class_Industry               0.380
status_2018_NOT_YET_RECRUITING            0.650
dtype: float64

## 11. Robustness check – trials already recruiting in 2018

Withdrawn trials come almost entirely from trials that were *not yet recruiting* in 2018 (see the README). To check that the model isn't only learning "not yet recruiting → withdrawn", Baseline 2 is scored again on the **recruiting trials only**. Their positive rate is different, so the comparison point is that subgroup's own rate.

In [21]:
recruiting = (X_valid["status_2018"] == "RECRUITING").to_numpy()

subgroup_result = evaluate(
    y_valid[recruiting], valid_prob_2[recruiting],
    "2 · all features, RECRUITING trials only",
)
display(pd.DataFrame([subgroup_result]).set_index("model").round(3))

,n_trials,positive_rate,pr_auc,roc_auc,brier
model,,,,,
"2 · all features, RECRUITING trials only",8050,0.173,0.283,0.662,0.136


## 12. Findings

| Model (validation, 9,157 trials, 20.2% stopped) | PR-AUC | ROC-AUC | Brier |
|---|---|---|---|
| 0 · no model | 0.202 | 0.500 | 0.161 |
| 1 · phase + sponsor | 0.248 | 0.582 | 0.159 |
| **2 · all 38 features** | **0.383** | **0.695** | **0.148** |

- **Sanity check passed:** the no-model baseline scores exactly the positive rate (PR-AUC 0.202) and ROC-AUC 0.5.
- **Real but moderate signal.** With all features, PR-AUC is **1.9× the no-skill level**. `phase` + sponsor type alone add little (0.248), so most of the signal comes from the other features.
- **No sign of leakage:** nothing scores suspiciously high.
- **Largest coefficients (association, not cause):**
  - towards *stopped early*: `NOT_YET_RECRUITING` in 2018, an industry lead sponsor, a US site, Phase 1/2, and registration long before the start date
  - towards *completed*: `RECRUITING` in 2018, behavioural interventions, U.S. Fed/NIH sponsors, health-services research, Early Phase 1, and more sites
- **Surprise:** `overdue_2018` leans towards *completed*. Worth checking in week 3 (e.g. the stop rate of overdue vs not-overdue trials).
- **Robustness:** on trials already recruiting in 2018, PR-AUC is 0.283 against a base rate of 0.173 (ROC-AUC 0.662). The model isn't only learning "not yet recruiting → withdrawn", but the signal is weaker there.

**The number to beat in week 3: PR-AUC 0.383 on validation.**